# Tostal — Train & Evaluate All Models

This notebook clones (or pulls) the repo, installs dependencies, and runs:
1. **Facies classifier** — XGBoost on FORCE 2020 well logs
2. **Segmenter** — DeepLabV3 on DCID drill core images
3. **Evaluate** — accuracy, F1, IoU, confusion matrix

Estimated runtime: ~5 min CPU (facies) + ~30 min GPU (segmenter)

In [ ]:
import os, sys, subprocess
from pathlib import Path

## 1. Clone or Pull the Repository

In [ ]:
REPO_URL = "https://github.com/beaglabs/tostal.git"

# Detect if we're already inside the repo
repo_root = Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / ".git").exists():
        break
    repo_root = repo_root.parent

if (repo_root / ".git").exists():
    print(f"Found existing repo at {repo_root}, pulling latest...")
    subprocess.run(["git", "pull"], cwd=repo_root, check=True)
else:
    repo_root = Path.home() / "tostal"
    if (repo_root / ".git").exists():
        print(f"Repo already exists at {repo_root}, pulling latest...")
        subprocess.run(["git", "pull"], cwd=repo_root, check=True)
    elif repo_root.exists():
        print(f"Directory {repo_root} exists but is not a git repo, removing...")
        import shutil
        shutil.rmtree(repo_root)
        subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)
    else:
        print(f"Cloning into {repo_root}...")
        subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)

print(f"Repo root: {repo_root}")

## 2. Install Dependencies

In [ ]:
requirements = repo_root / "api" / "requirements.txt"
if requirements.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)
else:
    subprocess.run([
        sys.executable, "-m", "pip", "install",
        "xgboost", "pykrige", "scikit-learn", "torch", "torchvision",
        "scipy", "datasets", "lasio", "openpyxl", "tqdm", "pillow",
    ], check=True)

print("Dependencies installed.")

## 3. Prepare FORCE 2020 Data

Ensure LAS files are properly extracted from the zenodo zip before training.

In [ ]:
%cd {repo_root}

# Force clean extraction of LAS files from zenodo zip
import shutil, zipfile
force2020_dir = Path("training/data/force2020")
las_dir = force2020_dir / "las"
processed_pt = force2020_dir / "processed.pt"

if processed_pt.exists():
    print(f"Removing stale cache: {processed_pt}")
    processed_pt.unlink()

if las_dir.exists():
    print(f"Removing stale extraction: {las_dir}")
    shutil.rmtree(las_dir)

las_zip = force2020_dir / "force2020_las.zip"
if las_zip.exists():
    print(f"Extracting {las_zip} -> {las_dir}")
    las_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(las_zip, "r") as zf:
        zf.extractall(las_dir)
    print("Extraction complete.")

%run training/scripts/train_xgboost_facies.py

## 4. Train Segmenter (DeepLabV3)

Downloads DCID drill core images from HuggingFace, fine-tunes DeepLabV3-ResNet50, and saves the model to `training/models/segmenter.pt`.

**⚠️ Requires GPU (T4/V100 recommended). Falls back to a placeholder on CPU.**

In [ ]:
%cd {repo_root}
%run training/scripts/train_segmenter.py

## 5. Evaluate Models

Evaluates the trained XGBoost classifier on the full FORCE 2020 dataset and prints accuracy, F1, and confusion matrix.

In [ ]:
%cd {repo_root}
%run training/scripts/evaluate.py

## 6. Verify Outputs

In [ ]:
models_dir = repo_root / "training" / "models"
for f in sorted(models_dir.glob("*")):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name:40s}  {size_mb:8.2f} MB")
print("\nDone.")